In [ ]:
%load_ext autoreload

%reset -f
import sys
from os.path import dirname, join as pjoin
from os import listdir
sys.path.append('C:\\Users\\maryj\\OneDrive\\Desktop\\LAB') # set local directory
import numpy as np
import pickle
import random
#import xarray as xr
import pandas as pd
import warnings
from IPython import embed

#import h5py
import scipy.io as sio
from scipy import stats
import statsmodels as sm # import statsmodels.api as 

from scipy.optimize import minimize, basinhopping, curve_fit
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from tqdm.auto import tqdm, trange

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
## Directories 
%autoreload

behavior_dir = 'Z:\\People\\Jeremy\\bpod_behav' # Where the behavioral data has been stored
results_dir = 'Z:\\People\\Jeremy\\Tina' ## Use this directory to load the .pkl info, so save it here
project_dir = 'Z:\\People\\Jeremy\\Tina' ## This is where we want to save
imaging_dir = pjoin(project_dir,'Imaging_Beh_Analysis') 

# Load Jeremy's lab data structure
list_path = pjoin(results_dir,"Imaging_Dataset_JL.mat")
data = sio.loadmat(list_path, struct_as_record=False, squeeze_me=True)
sessions = data["sessionarray_all"]
reload = True
rerun = True

UsageError: Line magic function `%autoreload` not found.


In [32]:
# skip running this cell if access to server is not available/ to save time
rows = []
if reload:    
    for sess in np.atleast_1d(sessions): ### This loads sessions 
        # keep only groups you want
        date  = str(sess.Date).strip()
        group = str(sess.Group).strip()  
        mouse = str(sess.Animal).strip() 
        

        # load per-session summary
        beh_path = pjoin(behavior_dir, mouse, 'Archive_sum_folder', f"{mouse}_Foraging_{date}_summary.mat") ### will needto skip animals SCXXX

        try:
            mat_contents = sio.loadmat(beh_path, struct_as_record=False, squeeze_me=True) ### Skips all files that start with SCXXX
        except FileNotFoundError:
            print(f"File not found: {beh_path}")
            continue

        
        summary = mat_contents["summary"] 
    
        # ensure 1-D numpy arrays, then cast to int and tolist()
        a_arr    = np.asarray(summary.a).ravel().astype(int)
        r_arr    = np.asarray(summary.R).ravel().astype(int)
        LR_Prob = np.array(summary.LR_Prob).ravel().astype(float)
        Reaction_time = np.array(summary.Reaction_time).ravel().astype(float)
        
        rows.append({
            "Date": date,
            "Group": group,
            "Mouse": mouse, 
            "a": a_arr,
            "R": r_arr, 
            "LR_Prob": LR_Prob,
            "Reaction_Time": Reaction_time
            
        })

     # build the dataframe in the exact column order you want
    imaging_beh_session_summary = pd.DataFrame(rows, columns=["Date","Group", "Mouse","a", "R", "LR_Prob", "Reaction_Time"])
    
    # for saving the dataframe as a pickle file
    save_path = pjoin(imaging_dir, "imaging_beh_sessions_dataframe.pkl")
    imaging_beh_session_summary.to_pickle(save_path)
    tqdm.pandas()
    temp_df = imaging_beh_session_summary# .iloc[2]    
    print(f"DataFrame saved to {save_path}")
else: 
    # Load session summary
    imaging_beh_session_summary = pd.read_pickle(pjoin(imaging_dir,'imaging_beh_sessions_dataframe.pkl'))
    tqdm.pandas()
    temp_df = imaging_beh_session_summary# .iloc[2]
    #temp_df_onerow = temp_df.iloc[0]

imaging_beh_session_summary.head()




File not found: Z:\People\Jeremy\bpod_behav\SC060\Archive_sum_folder\SC060_Foraging_231204_summary.mat
File not found: Z:\People\Jeremy\bpod_behav\SC060\Archive_sum_folder\SC060_Foraging_231220_summary.mat
File not found: Z:\People\Jeremy\bpod_behav\SC060\Archive_sum_folder\SC060_Foraging_240104_summary.mat
File not found: Z:\People\Jeremy\bpod_behav\SC060\Archive_sum_folder\SC060_Foraging_240107_summary.mat
File not found: Z:\People\Jeremy\bpod_behav\SC061\Archive_sum_folder\SC061_Foraging_240204_summary.mat
File not found: Z:\People\Jeremy\bpod_behav\SC061\Archive_sum_folder\SC061_Foraging_240207_summary.mat
File not found: Z:\People\Jeremy\bpod_behav\SC061\Archive_sum_folder\SC061_Foraging_240210_summary.mat
File not found: Z:\People\Jeremy\bpod_behav\SC061\Archive_sum_folder\SC061_Foraging_240223_summary.mat
File not found: Z:\People\Jeremy\bpod_behav\SC064\Archive_sum_folder\SC064_Foraging_240420_summary.mat
File not found: Z:\People\Jeremy\bpod_behav\SC064\Archive_sum_folder\SC06

,Date,Group,Mouse,a,R,LR_Prob,Reaction_Time
0,221118,RSCATN,JL025,"[1, 1, 1, 1, 2, 1, 1, 3, 2, 1, 3, 1, 1, 1, 1, ...","[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[2.8923159999999997, 2.8556999999999997, 2.417..."
1,221122,RSCATN,JL025,"[1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, ...","[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[2.495942, 2.971745, 2.942519, 2.820585, 2.385..."
2,221118,RSCATN,JL026,"[2, 1, 1, 1, 1, 1, 1, 2, 3, 2, 2, 2, 2, 4, 2, ...","[0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[3.1302950000000003, 3.152237, 3.305116, 3.044..."
3,221120,RSCATN,JL026,"[2, 3, 2, 2, 2, 2, 2, 2, 3, 2, 2, 2, 3, 2, 2, ...","[1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[3.3334740000000003, 0.411344, 3.146925, 2.889..."
4,221201,RSCATN,JL026,"[2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, ...","[1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[3.095911, 2.992583, 2.99742, 2.51994999999999..."


In [33]:
temp_df[:10][["a", "R"]].map(len)

,a,R
0,512,512
1,853,853
2,540,540
3,593,593
4,672,672
5,432,432
6,685,685
7,556,556
8,727,727
9,946,946


# This is where we start

In [9]:
# load .pkl file for when access to server is not available/ to save time
im_beh_pkl = pd.read_pickle( "imaging_beh_sessions_dataframe.pkl")
temp_df = pd.DataFrame(im_beh_pkl)
temp_df.head()

,Date,Group,Mouse,a,R,LR_Prob,Reaction_Time
0,221118,RSCATN,JL025,"[1, 1, 1, 1, 2, 1, 1, 3, 2, 1, 3, 1, 1, 1, 1, ...","[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[2.8923159999999997, 2.8556999999999997, 2.417..."
1,221122,RSCATN,JL025,"[1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, ...","[0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, ...","[2.495942, 2.971745, 2.942519, 2.820585, 2.385..."
2,221118,RSCATN,JL026,"[2, 1, 1, 1, 1, 1, 1, 2, 3, 2, 2, 2, 2, 4, 2, ...","[0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[3.1302950000000003, 3.152237, 3.305116, 3.044..."
3,221120,RSCATN,JL026,"[2, 3, 2, 2, 2, 2, 2, 2, 3, 2, 2, 2, 3, 2, 2, ...","[1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[3.3334740000000003, 0.411344, 3.146925, 2.889..."
4,221201,RSCATN,JL026,"[2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, ...","[1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, ...","[3.095911, 2.992583, 2.99742, 2.51994999999999..."


In [10]:
### Alarm Miss Rate Caluclation
am_rates = []
for idx, sess in temp_df.iterrows():
    
    a = sess["a"]
    R = sess["R"]
    n_engaged = len(a) # since unengaged mice were already pre-excluded before data was given
    mouse = sess["Mouse"]
    date = sess["Date"]
    group = sess["Group"]

    session_id = idx

    # Removes miss/alarms from a and R
    mask_choice = np.isin(a, [1, 2])

    R_wo_missalarm = R[mask_choice]
    a_wo_missalarm = a[mask_choice]

    num_choice = len(a_wo_missalarm)

    a_alarm = a[a == 3]
    a_miss = a[a == 4]
    alarm_rate = len(a_alarm) / n_engaged
    miss_rate =len(a_miss) / n_engaged

    am_rates.append({
       "Alarm rate" : alarm_rate, 
       "Miss rate" : miss_rate, 
       "Num choice" : num_choice,
       "n engaged" : n_engaged,
       "Date": date, 
       "Mouse": mouse,
       "Session_ID": session_id,
        "Group": group

    })

# Print sanity check of final session
print("Recorded trials:", n_engaged)
print("Choice trials:", num_choice)
print("Alarm + Miss rate:", 100-(num_choice*100/n_engaged), "%")
print("Alarm rate:", alarm_rate*100, "%")
print("Miss rate:", miss_rate*100, "%")

Recorded trials: 551
Choice trials: 450
Alarm + Miss rate: 18.330308529945555 %
Alarm rate: 14.519056261343014 %
Miss rate: 3.8112522686025407 %


In [11]:
# Alarm Miss Rate Intermediate Dataframe

# am_rates list to df
am_rate_df = pd.DataFrame(am_rates)
am_rate_df

,Alarm rate,Miss rate,Num choice,n engaged,Date,Mouse,Session_ID,Group
0,0.017578,0.015625,495,512,221118,JL025,0,RSCATN
1,0.028136,0.002345,827,853,221122,JL025,1,RSCATN
2,0.090741,0.057407,460,540,221118,JL026,2,RSCATN
3,0.094435,0.042159,512,593,221120,JL026,3,RSCATN
4,0.119048,0.005952,588,672,221201,JL026,4,RSCATN
...,...,...,...,...,...,...,...,...
113,0.034557,0.008639,443,463,241111,JL401,113,Scnn1a
114,0.054920,0.045767,393,437,241110,JL403,114,Scnn1a
115,0.012090,0.018998,561,579,241202,JL403,115,Scnn1a
116,0.060362,0.056338,439,497,241204,JL403,116,Scnn1a


In [12]:
# Win Stay Lose Switch
from numpy import diff

p_norm_rates = []
for idx, sess in temp_df.iterrows():
    
    # Columns
    a = sess["a"]
    R = sess["R"]
    mouse = sess["Mouse"]
    date = sess["Date"]
    group = sess["Group"]
    session_id = idx

    All_Trans = len(a_wo_missalarm) - 1 # number of possible switch or stay
    N_Switch = sum(abs(diff(a_wo_missalarm)))
    N_Stay = (All_Trans - N_Switch)
    N_Win = sum(R_wo_missalarm[:-1]) # R_wo_missalarm(1:end-1)
    N_Lose = All_Trans-N_Win

    N_WinSwitch = sum(
        (abs(diff(a_wo_missalarm)) > 0) & 
        (R_wo_missalarm[:-1].astype(bool)) 
    )

    N_LoseSwitch = sum(
    (abs(diff(a_wo_missalarm)) > 0) &
    (~R_wo_missalarm[:-1].astype(bool))
    )

    P_WinSwitch = N_WinSwitch/N_Win
    P_WinStay = 1 - P_WinSwitch # Output
    P_LoseSwitch = N_LoseSwitch / N_Lose # Output
    P_LoseStay = 1 - P_LoseSwitch
    P_Stay = N_Stay / All_Trans # calculates N
    P_Switch = N_Switch / All_Trans

    Normalized_WinStay = P_WinStay / P_Stay #Output
    Normalized_LoseSwitch = P_LoseSwitch / P_Switch # Output

    p_norm_rates.append({
        # "All_Trans": All_Trans,
        # "N_Switch": N_Switch,
        # "N_Stay": N_Stay,
        # "N_Win": N_Win,
        # "N_Lose": N_Lose,
        # "N_WinSwitch": N_WinSwitch,
        # "N_LoseSwitch": N_LoseSwitch,
        # "P_WinSwitch": P_WinSwitch,
        "P_WinStay": P_WinStay,
        "P_LoseSwitch": P_LoseSwitch,
        # "P_LoseStay": P_LoseStay,
        # "P_Stay": P_Stay,
        # "P_Switch": P_Switch,
        "Normalized_WinStay": Normalized_WinStay,
        "Normalized_LoseSwitch": Normalized_LoseSwitch,
        "Date": date, 
        "Mouse": mouse,
        "Session_ID": session_id,
        "Group": group
        })

# P_WinStay = p_norm_rates[0]['P_WinStay'] 
# P_LoseSwitch = p_norm_rates[0]['P_LoseSwitch'] 
# Normalized_WinStay = p_norm_rates[0]['Normalized_WinStay'] 
# Normalized_LoseSwitch = p_norm_rates[0]['Normalized_LoseSwitch'] 

# Print sanity check of final session
# print (f"P_WinStay: {P_WinStay}")
# print (f"P_LoseSwitch: {P_LoseSwitch}")
# print (f"Normalized_WinStay: {Normalized_WinStay}")
# print (f"Normalized_LoseSwitch: {Normalized_LoseSwitch}")


In [43]:
p_norm_rates_df = pd.DataFrame(p_norm_rates)
p_norm_rates_df

,P_WinStay,P_LoseSwitch,Normalized_WinStay,Normalized_LoseSwitch,Date,Mouse,Session_ID,Group
0,0.807175,0.283186,1.059712,1.188322,221118,JL025,0,RSCATN
1,0.807175,0.283186,1.059712,1.188322,221122,JL025,1,RSCATN
2,0.807175,0.283186,1.059712,1.188322,221118,JL026,2,RSCATN
3,0.807175,0.283186,1.059712,1.188322,221120,JL026,3,RSCATN
4,0.807175,0.283186,1.059712,1.188322,221201,JL026,4,RSCATN
...,...,...,...,...,...,...,...,...
113,0.807175,0.283186,1.059712,1.188322,241111,JL401,113,Scnn1a
114,0.807175,0.283186,1.059712,1.188322,241110,JL403,114,Scnn1a
115,0.807175,0.283186,1.059712,1.188322,241202,JL403,115,Scnn1a
116,0.807175,0.283186,1.059712,1.188322,241204,JL403,116,Scnn1a


In [62]:
am_pnorm_df = pd.merge(am_rate_df , p_norm_rates_df, on=["Session_ID", "Date", "Group", "Mouse"])

# for saving the dataframe as a pickle file
save_path = pjoin(imaging_dir, "AlarmMiss_PNorm_Rates.pkl")
am_pnorm_df.to_pickle(save_path)
tqdm.pandas()
print(f"DataFrame saved to {save_path}")
am_pnorm_df.head()

DataFrame saved to Z:\People\Jeremy\Tina\Imaging_Beh_Analysis\AlarmMiss_PNorm_Rates.pkl


,Alarm rate,Miss rate,Num choice,n engaged,Date,Mouse,Session_ID,Group,P_WinStay,P_LoseSwitch,Normalized_WinStay,Normalized_LoseSwitch
0,0.017578,0.015625,495,512,221118,JL025,0,RSCATN,0.807175,0.283186,1.059712,1.188322
1,0.028136,0.002345,827,853,221122,JL025,1,RSCATN,0.807175,0.283186,1.059712,1.188322
2,0.090741,0.057407,460,540,221118,JL026,2,RSCATN,0.807175,0.283186,1.059712,1.188322
3,0.094435,0.042159,512,593,221120,JL026,3,RSCATN,0.807175,0.283186,1.059712,1.188322
4,0.119048,0.005952,588,672,221201,JL026,4,RSCATN,0.807175,0.283186,1.059712,1.188322


In [63]:
# For loading alarm/miss + P rates dataframe file 
am_pnorm_pkl = pd.read_pickle(pjoin(imaging_dir, "AlarmMiss_PNorm_Rates.pkl"))
temp_pkl_df = pd.DataFrame(am_pnorm_pkl)
temp_pkl_df

,Alarm rate,Miss rate,Num choice,n engaged,Date,Mouse,Session_ID,Group,P_WinStay,P_LoseSwitch,Normalized_WinStay,Normalized_LoseSwitch
0,0.017578,0.015625,495,512,221118,JL025,0,RSCATN,0.807175,0.283186,1.059712,1.188322
1,0.028136,0.002345,827,853,221122,JL025,1,RSCATN,0.807175,0.283186,1.059712,1.188322
2,0.090741,0.057407,460,540,221118,JL026,2,RSCATN,0.807175,0.283186,1.059712,1.188322
3,0.094435,0.042159,512,593,221120,JL026,3,RSCATN,0.807175,0.283186,1.059712,1.188322
4,0.119048,0.005952,588,672,221201,JL026,4,RSCATN,0.807175,0.283186,1.059712,1.188322
...,...,...,...,...,...,...,...,...,...,...,...,...
113,0.034557,0.008639,443,463,241111,JL401,113,Scnn1a,0.807175,0.283186,1.059712,1.188322
114,0.054920,0.045767,393,437,241110,JL403,114,Scnn1a,0.807175,0.283186,1.059712,1.188322
115,0.012090,0.018998,561,579,241202,JL403,115,Scnn1a,0.807175,0.283186,1.059712,1.188322
116,0.060362,0.056338,439,497,241204,JL403,116,Scnn1a,0.807175,0.283186,1.059712,1.188322


In [ ]:
### 2-fold Cross-Validation predicted accuracy 
print("Running prediction accuracy...")

N = len(temp_df["a"]) 

# Exclude first  for training/testing
usable_trials = N - 10 
half = usable_trials // 2 

## Then split the trials to 1st and 2nd equal halves, training and testing
id_1 = np.arange(10, 10 + half)
id_2 = np.arange(10 + half, N)

n_choice = 2 # L & R lickports
# Verifies that fold split was successful, prints out first/last 5 trial indicies...
print("Fold 1 split:", id_1[:5], id_1[-5:], len(id_1)) # ..and total number of trials in each fold 
print("Fold 2 split:", id_2[:5], id_2[-5:], len(id_2)) 
                               
# Values expected scratch work
# 118 / 2 = 59, 
# 118 - 10 = 108/2 = 54

#### Full model 
# params : 5x1 vector, alpha_rew, alpha_unrew, beta0, beta1, decay
# alpha_rew : learning rate on rewarded trials 
# alpha_unrew : learning rate on unrewarded trials
# beta0 : bias term 
# beta1: sesitivity to the diff between QL and QR 
# decay : forgetting rate for unchosen, Qunchosen(t+1) = (1-decay)*Qunchosen(t)

## Loop through sessions
input = []
for _, sess in temp_df.iterrows(): 
    a = sess["a"]
    R = sess["R"] 
    n_choice = 2
    n_trials = len(a)
    LR_Prob = sess["LR_Prob"]
    Reaction_time = sess["Reaction_Time"]

    ### Fit full model to all data to get parameters.
    # helper
    from scipy.optimize import minimize 
    def fit_params_mle(fun, x0, lb, ub): 
        bounds = list(zip(lb, ub))
        res = minimize(fun, x0=np.array(x0, dtype=float), 
                    bounds=bounds, method="L-BFGS-B", 
                    options={"disp":False})
        return res.x
            
    ## Create 2 functions - negative log-likelihood for each half of the data (fold objective funcntions)    
    fun_full_1 = lambda params: RW_fitmodel_bias_decay_nloglik(
        params, n_choice, len(id_1), a[id_1], R[id_1]
    )
    fun_full_2 = lambda params: RW_fitmodel_bias_decay_nloglik(
        params, n_choice, len(id_2), a[id_2], R[id_2]
    )

    # init + bounds
    params_x0_full = [0.2, 0.2, 3.0, 0.0, 0.0]
    lb_full = [0.0, 0.0, -15.0, -10.0, 0.0]
    ub_full= [1.0, 1.0,  15.0,  10.0, 1.0]
    
    params_1 = fit_params_mle(fun_full_1, params_x0_full, lb_full, ub_full)
    params_2 = fit_params_mle(fun_full_2, params_x0_full, lb_full, ub_full)
    # cross-validated prediction
    _, pout_2 = RW_fitmodel_bias_decay(params_1, n_choice, len(id_2), a[id_2], R[id_2])
    _, pout_1 = RW_fitmodel_bias_decay(params_2, n_choice, len(id_1), a[id_1], R[id_1])
    # RW dependencies

#
    def RW_fitmodel_bias_decay(params, n_choice, n_trials, a, R, LR_Prob=None, Reaction_time=None): 
        alpha_rew, alpha_unrew, beta1, beta0, decay = params
    
        a = np.asarray(a).astype(int)
        R = np.asarray(R).astype(float)
        
        q = np.array([0.5, 0.5], dtype=float) # probs are equally likely before softmax
        QQ = np.full((n_trials, n_choice), np.nan, dtype=float) # Output 
        PP = np.full((n_trials, n_choice), np.nan, dtype=float) # Output


        for t in range(n_trials): 
            logits = np.array([0.0, beta0]) + beta1 * q
            ev = np.exp(logits) # softmax
            p = ev / ev.sum() # probability each choice

            QQ[t,:] = q 
            PP[t,:] = p 

            # Value Update Rules
            if a[t] == 1: # choice 'Right'
                lr = alpha_rew if R[t] == 1 else alpha_unrew
                q[0] = q[0] + lr * (R[t] - q[0]) #
                q[1] = (1 - decay) * q[1]

            elif a[t] == 2:# choice 'Left'
                q[0] = (1 - decay) * q[0]
                lr = alpha_rew if R[t] == 1 else alpha_unrew
                q[1] = q[1] + lr * (R[t] - q[1])

            elif a[t] in (3, 4): # choice 'Miss/Alarm'
                q[:] = (1 - decay) * q

        ## Compute likelihood: sum of logs of p(observed choice) for all trials
        # matlab: loglik = sum(log(PP(a==1,1))) + sum(log(PP(a==2,2))), nloglik = -loglik
        # Convert MATLAB 1/2 choice codes to Python row selection
        mask1 = (a == 1)
        mask2 = (a == 2)
        # small safety: avoid log(0)
        # in practice, softmax won't produce 0, but w large dataset, this is just in case
        # eps = 1e-12
        loglik = np.sum(np.log(PP[mask1, 0])) + np.sum(np.log(PP[mask2, 1]))
        #nloglik = -1 *loglik
        # Output 
        pout = {"QQ": QQ, "PP": PP, "loglik": loglik, "params": np.array(params, dtype=float)}
        return loglik, pout

       

    def RW_fitmodel_bias_decay_nloglik(params, n_choice, n_trials, a, R, LR_Prob=None, Reaction_time=None):
        loglik, _ = RW_fitmodel_bias_decay(params, n_choice, n_trials, a, R, LR_Prob, Reaction_time)
        return -loglik

        
  

input.append({
    "total trials": n_trials,
    "choice trials": n_choice,
    "a": a, 
    "R": R,
    "LR Prob": LR_Prob,
    "Reaction time ": Reaction_time,
    "choice trials fold1" : np.sum((a[id_1]==1) | (a[id_1]==2)),
    "choice trials fold2": np.sum((a[id_2]==1) | (a[id_2]==2))
}) 
#   prediction accuracy helper
def prediction_accuracy(PP_col1, choices):

    # choices are coded 1/2 like MATLAB; keep only those
    valid = (choices == 1) | (choices == 2) 
    if np.sum(valid) ==0: 
        return np.nan
    p = PP_col1[valid]
    c = choices[valid]

    pred = np.where(p > 0.5, 1, 2)   # if P(choice1)>0.5 predict 1 else 2
    return np.mean(pred == c)

# Full Model Output
PA_RL_full_2 = prediction_accuracy(pout_2["PP"][:, 0], a[id_2])
PA_RL_full_1 = prediction_accuracy(pout_1["PP"][:, 0], a[id_1])
PA_RL_full_mean = 0.5 * (PA_RL_full_2 + PA_RL_full_1)



print("Full model PA:", PA_RL_full_mean)
print("choice trials fold1")
print("choice trials fold2")                                                

print(input)

Running prediction accuracy...
Fold 1 split: [10 11 12 13 14] [59 60 61 62 63] 54
Fold 2 split: [64 65 66 67 68] [113 114 115 116 117] 54


NameError: name 'params_1' is not defined

: 

In [ ]:
# Full Model
    
results = []
for _, sess in temp_df.iterrows(): 
    a = sess["a"]
    R = sess["R"] 

    n_choice = 2
    n_trials = len(a)

    LR_Prob = sess["LR_Prob"]
    Reaction_time = sess["Reaction_Time"]

    # removes miss/alarms from a and R
    R_wo_missalarm = R[(a == 1) | (a == 2)]
    a_wo_missalarm = a[(a == 1) | (a == 2)]
    num_choice = len(a_wo_missalarm)
    n_trials = len(a)

### 



    results.append({
        "n_trials" : n_trials,
        "Choice Trials" : num_choice*100/n_engaged,
        "Engaged Trials" : n_engaged,
        "Alarm + Miss Rate" : 100-(num_choice*100/n_engaged),
        "Alarm rate" : alarm_rate* 100,
        "Miss rate" : miss_rate*100,
        
    })

results_df = pd.DataFrame(results)


print(results_df)
print("Full model PA:", PA_full_mean)

C:\Users\maryj\AppData\Local\Temp\ipykernel_10728\3276617833.py:13: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(fun, x0=np.array(params_x0_full, dtype=float),


     n_trials  Choice Trials  Engaged Trials  Alarm + Miss Rate  Alarm rate  \
0         512     419.491525             118        -319.491525    7.627119   
1         853     700.847458             118        -600.847458   20.338983   
2         540     389.830508             118        -289.830508   41.525424   
3         593     433.898305             118        -333.898305   47.457627   
4         672     498.305085             118        -398.305085   67.796610   
..        ...            ...             ...                ...         ...   
113       463     375.423729             118        -275.423729   13.559322   
114       437     333.050847             118        -233.050847   20.338983   
115       579     475.423729             118        -375.423729    5.932203   
116       497     372.033898             118        -272.033898   25.423729   
117       551     381.355932             118        -281.355932   67.796610   

     Miss rate  PA training  PA testing  Choice tri

In [ ]:
## Questions 

# Q1 Does the matlab NLL function for the full model ignore non-choice trials by default? 
# A: Yes
#-- need to input raw vector that has not removed miss/alarm 
# -- L2 reg rile has a (inclusive, but for a lot of analysis we exclude it) ignore params from fun full 1, parameters actually come from RWfit model files 

# Q2 Should I use the 1-D a_clean and R_clean here, or can I plug in a_wo_missalarm and R_wo_missalarm? 
# --> Why does excluding non 1&2 values inside the CVPA (assigning them to folds) change the analysis? 
    # ChatGPT: ...But for CVPA, if you define folds on a_wo_missalarm, you are subtly changing the fold boundary compared to MATLAB 
    # (because you removed trials before splitting). That’s not “wrong,” but it’s not the same analysis anymore.

# --- -solutino: for loop to give function one value at a time solves this problem 




